<a href="https://colab.research.google.com/github/AhnafRifqi/Algoritma-CaseMethod2/blob/main/DM_Jobsheet11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib
matplotlib.use('Agg') # Gunakan backend Agg untuk menyimpan plot tanpa menampilkannya secara interaktif
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from io import BytesIO # Digunakan untuk menyimpan plot
import base64 # Digunakan untuk encode plot ke base64 jika diperlukan (misal untuk display dalam HTML/Markdown)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataSiswa = pd.read_csv('/content/drive/MyDrive/Data_Mining/Dataset/jobsheet_11/Dataset_SIB_2G.csv', sep=',')
print(dataSiswa.head())

                 Timestamp                            Nama Tempat Lahir  \
0  2026-02-10 18:47:53.126           Pratama Angga Saputra       Malang   
1  2026-02-10 18:55:03.138  Muchammad Nabil Haykal Widarto  Kota Malang   
2  2026-02-10 18:55:50.648              Moh. Rafael Abrari     Lamongan   
3  2026-02-10 19:49:43.557          Muhamad Miftahul Akmal      Cilegon   
4  2026-02-10 19:54:35.795     Narendra Daniswara Alamsyah   Bojonegoro   

  Tanggal Lahir Jenis Kelamin  Tinggi Badan (cm)  Berat Badan (kg)  \
0    2005-07-10     Laki-Laki                165              52.5   
1    2004-03-22     Laki-Laki                175             123.0   
2    2006-11-27     Laki-Laki                173              55.0   
3    2005-03-05     Laki-Laki                168              70.0   
4    2006-02-16     Laki-Laki                166              64.0   

     Kota Asal                Alamat Domisili Saat Ini Jenis Kendaraan Utama  \
0       Malang      Jl. Mergan lori 2b no 51 RT 

### PERCOBAAN 1 - Decision Tree Dataset Sederhana BTS

In [ ]:
# Penjelasan: Bagian ini mendefinisikan dataset sederhana 'BTS', melatih Decision Tree,
# memprediksi data baru, dan memvisualisasikan pohon keputusan.

print("="*50)
print("PERCOBAAN 1: Decision Tree Dataset Sederhana BTS")
print("="*50)

# 1. Mendefinisikan dataset BTS dalam bentuk array
# Suka Cilok, Suka Kopi, Usia, Suka BTS (Target)
# Yes(1)/No(0)
data_bts = np.array([
    [1, 1, 7, 0],  # No Suka BTS
    [1, 0, 12, 0], # No Suka BTS
    [0, 1, 18, 1], # Yes Suka BTS
    [0, 1, 35, 1], # Yes Suka BTS
    [1, 1, 38, 1], # Yes Suka BTS
    [1, 0, 50, 0], # No Suka BTS
    [0, 0, 83, 0]  # No Suka BTS
])

feature_names_bts = ['Suka Cilok', 'Suka Kopi', 'Usia']
class_names_bts = ['Tidak Suka BTS', 'Suka BTS']

X_bts = data_bts[:, :3]  # Fitur: Suka Cilok, Suka Kopi, Usia
y_bts = data_bts[:, 3]   # Target: Suka BTS

# 2. Melatih DecisionTreeClassifier
dt_bts = DecisionTreeClassifier(random_state=42)
dt_bts.fit(X_bts, y_bts)

print("\n--- Model Training Selesai ---")

# 3. Memprediksi data baru: [Suka Cilok=Yes(1), Suka Kopi=Yes(1), Usia=15]
new_data_bts = np.array([[1, 1, 15]])
prediction_bts = dt_bts.predict(new_data_bts)

# 4. Mencetak hasil prediksi (angka dan labelnya)
predicted_label_bts = class_names_bts[prediction_bts[0]]
print(f"\nHasil Prediksi untuk data baru [Suka Cilok=Yes(1), Suka Kopi=Yes(1), Usia=15]: {prediction_bts[0]} ({predicted_label_bts})")

# Output yang diharapkan: prediksi = [1] artinya "Suka BTS"
print("Prediksi yang diharapkan: [1] (Suka BTS)")

# 5. Memvisualisasikan Decision Tree dan menyimpan sebagai gambar PNG
plt.figure(figsize=(12, 8))
plot_tree(dt_bts, feature_names=feature_names_bts, class_names=class_names_bts,
          rounded=True, filled=True, fontsize=10)
plt.title("Decision Tree untuk Dataset BTS Sederhana")
plt.savefig('decision_tree_bts.png')
plt.close()
print("Visualisasi Decision Tree BTS telah disimpan sebagai 'decision_tree_bts.png'")

PERCOBAAN 1: Decision Tree Dataset Sederhana BTS

--- Model Training Selesai ---

Hasil Prediksi untuk data baru [Suka Cilok=Yes(1), Suka Kopi=Yes(1), Usia=15]: 1 (Suka BTS)
Prediksi yang diharapkan: [1] (Suka BTS)
Visualisasi Decision Tree BTS telah disimpan sebagai 'decision_tree_bts.png'


### PERCOBAAN 2 - Decision Tree Dataset Iris

In [ ]:
# Penjelasan: Bagian ini memuat dataset Iris, memilih fitur tertentu, membagi data,
# melatih Decision Tree, mengevaluasi model, dan memvisualisasikan pohon keputusan.

print("\n" + "="*50)
print("PERCOBAAN 2: Decision Tree Dataset Iris")
print("="*50)

# 1. Load dataset Iris dari sklearn.datasets
from sklearn.datasets import load_iris

iris = load_iris()
df_iris = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target
df_iris['target_names'] = df_iris['target'].apply(lambda x: iris.target_names[x])

# 2. Gunakan hanya fitur petal length (cm) dan petal width (cm) (kolom index 2 dan 3)
X_iris = iris.data[:, 2:]  # Petal length and petal width
y_iris = iris.target

# 3. Split data: 80% train, 20% test (random_state=42)
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

print("\n--- Data Iris telah dibagi: 80% Train, 20% Test ---")

# 4. Latih DecisionTreeClassifier(random_state=42)
dt_iris = DecisionTreeClassifier(random_state=42)
dt_iris.fit(X_train_iris, y_train_iris)

print("--- Model Decision Tree Iris telah dilatih ---")

# 5. Prediksi dan hitung akurasi menggunakan accuracy_score
y_pred_iris = dt_iris.predict(X_test_iris)
accuracy_iris = accuracy_score(y_test_iris, y_pred_iris)
print(f"\nAkurasi Model Decision Tree Iris: {accuracy_iris:.2f}")

# 6. Cetak classification_report dengan target_names=iris.target_names
print("\nClassification Report untuk Decision Tree Iris:")
print(classification_report(y_test_iris, y_pred_iris, target_names=iris.target_names))

# 7. Visualisasikan Decision Tree dan simpan sebagai PNG
plt.figure(figsize=(15, 10))
plot_tree(dt_iris, feature_names=iris.feature_names[2:], class_names=iris.target_names,
          rounded=True, filled=True, fontsize=10)
plt.title("Decision Tree untuk Dataset Iris (Petal Length & Width)")
plt.savefig('decision_tree_iris.png')
plt.close()
print("Visualisasi Decision Tree Iris telah disimpan sebagai 'decision_tree_iris.png'")


PERCOBAAN 2: Decision Tree Dataset Iris

--- Data Iris telah dibagi: 80% Train, 20% Test ---
--- Model Decision Tree Iris telah dilatih ---

Akurasi Model Decision Tree Iris: 1.00

Classification Report untuk Decision Tree Iris:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Visualisasi Decision Tree Iris telah disimpan sebagai 'decision_tree_iris.png'


### PERCOBAAN 3 - KNN Dataset Iris

In [ ]:
# Penjelasan: Bagian ini melatih model KNN pada dataset Iris untuk menjawab pertanyaan
# tentang parameter default KNN di sklearn, sesuai permintaan pengguna.

print("\n" + "="*50)
print("PERCOBAAN 3: KNN Dataset Iris")
print("="*50)


from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data
y = iris.target
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=.1)
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier()
classifier.fit(X_train, y_train)
predictions = classifier.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_value = accuracy_score(y_test, predictions)
print(f"\nAkurasi model KNN (dari kode pengguna): {accuracy_value:.2f}")

print("\n--- Analisis Parameter Default KNeighborsClassifier ---")

# 5. Cetak parameter default yang digunakan: n_neighbors, metric, p
print("\nParameter default KNeighborsClassifier:")
print(f"  n_neighbors (K): {classifier.n_neighbors}")
print(f"  metric: {classifier.metric}")
print(f"  p (untuk Minkowski/Euclidean): {classifier.p}")

# 6. Jawab pertanyaan: "Berapa K dan metode distance apa yang digunakan secara default?"
print("\nJawaban pertanyaan modul:")
print(f"Secara default, `K` yang digunakan adalah {classifier.n_neighbors}.")
print(f"Metode distance yang digunakan adalah `{classifier.metric}` distance, dengan parameter `p={classifier.p}`.")
print("Untuk `p=2` pada metrik Minkowski, ini setara dengan Euclidean distance.")
print("Jadi, K=5 dan Minkowski distance (p=2 = Euclidean) digunakan secara default.")


PERCOBAAN 3: KNN Dataset Iris

Akurasi model KNN (dari kode pengguna): 0.93

--- Analisis Parameter Default KNeighborsClassifier ---

Parameter default KNeighborsClassifier:
  n_neighbors (K): 5
  metric: minkowski
  p (untuk Minkowski/Euclidean): 2

Jawaban pertanyaan modul:
Secara default, `K` yang digunakan adalah 5.
Metode distance yang digunakan adalah `minkowski` distance, dengan parameter `p=2`.
Untuk `p=2` pada metrik Minkowski, ini setara dengan Euclidean distance.
Jadi, K=5 dan Minkowski distance (p=2 = Euclidean) digunakan secara default.


### LATIHAN - Decision Tree & KNN pada Dataset Siswa

#### Bagian A: Decision Tree pada Dataset Siswa (`dataSiswa`)

In [ ]:
# Melatih Decision Tree untuk memprediksi `Is_Normal_BMI`, dan mengevaluasinya.

print("\n" + "="*50)
print("LATIHAN BAGIAN A: Decision Tree pada Dataset Siswa (dataSiswa)")
print("="*50)

# --- Code to define X_dataSiswa and y_dataSiswa (previously in deleted cell 0a3ebb20) ---
# 1. Konversi 'Tinggi Badan (cm)' ke meter
dataSiswa['Tinggi Badan (m)'] = dataSiswa['Tinggi Badan (cm)'] / 100

# 2. Hitung BMI (Body Mass Index)
# Formula: Berat (kg) / (Tinggi (m))^2
dataSiswa['BMI'] = dataSiswa['Berat Badan (kg)'] / (dataSiswa['Tinggi Badan (m)'])**2

# 3. Kategorikan BMI
# Standar WHO: <18.5 (Kurus), 18.5-24.9 (Normal), 25-29.9 (Gemuk), >=30 (Obesitas)
def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Kurus'
    elif 18.5 <= bmi < 24.9:
        return 'Normal'
    elif 25 <= bmi < 29.9:
        return 'Gemuk'
    else:
        return 'Obesitas'

dataSiswa['Kategori_BMI'] = dataSiswa['BMI'].apply(categorize_bmi)

# 4. Definisikan target biner 'Is_Normal_BMI'
# 1 jika BMI Normal, 0 jika tidak (Kurus, Gemuk, Obesitas)
dataSiswa['Is_Normal_BMI'] = (dataSiswa['Kategori_BMI'] == 'Normal').astype(int)

# Fitur yang akan digunakan untuk model (termasuk Usia, Tinggi, Berat, BMI)
# Usia (tahun) juga merupakan fitur penting yang diminta user
features_dataSiswa = ['Usia (tahun)', 'Tinggi Badan (cm)', 'Berat Badan (kg)', 'BMI']
target_dataSiswa = 'Is_Normal_BMI'

X_dataSiswa = dataSiswa[features_dataSiswa]
y_dataSiswa = dataSiswa[target_dataSiswa]
# --- End of added code ---

# Split 80% train, 20% test (random_state=42)
X_train_dt_siswa, X_test_dt_siswa, y_train_dt_siswa, y_test_dt_siswa = train_test_split(
    X_dataSiswa, y_dataSiswa, test_size=0.2, random_state=42, stratify=y_dataSiswa # Stratify untuk menjaga proporsi kelas target
)

print("\n--- Data Siswa telah dibagi: 80% Train, 20% Test ---")

# Latih DecisionTreeClassifier (random_state=42)
dt_siswa_real = DecisionTreeClassifier(random_state=42)
dt_siswa_real.fit(X_train_dt_siswa, y_train_dt_siswa)

print("--- Model Decision Tree Siswa telah dilatih ---")

# Hitung dan cetak akurasi + classification_report
y_pred_dt_siswa_real = dt_siswa_real.predict(X_test_dt_siswa)
accuracy_dt_siswa_real = accuracy_score(y_test_dt_siswa, y_pred_dt_siswa_real)
print(f"\nAkurasi Decision Tree pada Dataset Siswa (Is_Normal_BMI): {accuracy_dt_siswa_real:.2f}")

print("\nClassification Report Decision Tree Siswa:")
print(classification_report(y_test_dt_siswa, y_pred_dt_siswa_real, target_names=['BMI Tidak Normal', 'BMI Normal']))

# Visualisasikan Decision Tree
plt.figure(figsize=(20, 15))
plot_tree(dt_siswa_real, feature_names=features_dataSiswa, class_names=['BMI Tidak Normal', 'BMI Normal'],
          rounded=True, filled=True, fontsize=10)
plt.title("Decision Tree untuk Prediksi Is_Normal_BMI (dataSiswa)")
plt.savefig('decision_tree_dataSiswa_bmi.png')
plt.close()
print("Visualisasi Decision Tree dataSiswa telah disimpan sebagai 'decision_tree_dataSiswa_bmi.png'")


LATIHAN BAGIAN A: Decision Tree pada Dataset Siswa (dataSiswa)

--- Data Siswa telah dibagi: 80% Train, 20% Test ---
--- Model Decision Tree Siswa telah dilatih ---

Akurasi Decision Tree pada Dataset Siswa (Is_Normal_BMI): 1.00

Classification Report Decision Tree Siswa:
                  precision    recall  f1-score   support

BMI Tidak Normal       1.00      1.00      1.00         2
      BMI Normal       1.00      1.00      1.00         2

        accuracy                           1.00         4
       macro avg       1.00      1.00      1.00         4
    weighted avg       1.00      1.00      1.00         4

Visualisasi Decision Tree dataSiswa telah disimpan sebagai 'decision_tree_dataSiswa_bmi.png'


#### Bagian B: KNN dengan Berbagai Nilai K pada Dataset Siswa (`dataSiswa`)

In [ ]:
# Penjelasan: Bagian ini menerapkan KNN dengan berbagai nilai K dan dua metrik distance (Manhattan & Euclidean)
# pada dataset `dataSiswa` setelah normalisasi, lalu membandingkan akurasinya.

print("\n" + "="*50)
print("LATIHAN BARU BAGIAN B: KNN dengan Berbagai Nilai K (dataSiswa)")
print("="*50)

# Gunakan dataset siswa yang sama dan bagi data lagi
X_train_knn_siswa_real, X_test_knn_siswa_real, y_train_knn_siswa_real, y_test_knn_siswa_real = train_test_split(
    X_dataSiswa, y_dataSiswa, test_size=0.2, random_state=42, stratify=y_dataSiswa
)

# Lakukan normalisasi dengan StandardScaler
scaler_siswa_real = StandardScaler()
X_train_knn_siswa_real_scaled = scaler_siswa_real.fit_transform(X_train_knn_siswa_real)
X_test_knn_siswa_real_scaled = scaler_siswa_real.transform(X_test_knn_siswa_real)

print("--- Data Siswa telah dinormalisasi untuk KNN ---")

# Uji nilai K: [1, 3, 5, 7, 9, 11, 13, 15]
# FIX: Ensure k_values do not exceed the number of training samples.
# len(X_train_knn_siswa_real) is 13, so max K should be 13.
k_values_real = [1, 3, 5, 7, 9, 11, 13]
results_real = {
    'K': [],
    'Minkowski (p=1)': [], # Manhattan
    'Euclidean (p=2)': []
}

best_k_manhattan_real = -1
best_accuracy_manhattan_real = -1.0
best_k_euclidean_real = -1
best_accuracy_euclidean_real = -1.0

print("\nMelatih model KNN dengan berbagai nilai K...")
for k in k_values_real:
    results_real['K'].append(k)

    # KNN dengan Minkowski distance (p=1) -> Manhattan
    knn_manhattan_real = KNeighborsClassifier(n_neighbors=k, p=1) # p=1 for Manhattan
    knn_manhattan_real.fit(X_train_knn_siswa_real_scaled, y_train_knn_siswa_real)
    y_pred_manhattan_real = knn_manhattan_real.predict(X_test_knn_siswa_real_scaled)
    accuracy_manhattan_real = accuracy_score(y_test_knn_siswa_real, y_pred_manhattan_real)
    results_real['Minkowski (p=1)'].append(accuracy_manhattan_real * 100)

    if accuracy_manhattan_real > best_accuracy_manhattan_real:
        best_accuracy_manhattan_real = accuracy_manhattan_real
        best_k_manhattan_real = k

    # KNN dengan Euclidean distance (p=2)
    knn_euclidean_real = KNeighborsClassifier(n_neighbors=k, p=2) # p=2 for Euclidean
    knn_euclidean_real.fit(X_train_knn_siswa_real_scaled, y_train_knn_siswa_real)
    y_pred_euclidean_real = knn_euclidean_real.predict(X_test_knn_siswa_real_scaled)
    accuracy_euclidean_real = accuracy_score(y_test_knn_siswa_real, y_pred_euclidean_real)
    results_real['Euclidean (p=2)'].append(accuracy_euclidean_real * 100)

    if accuracy_euclidean_real > best_accuracy_euclidean_real:
        best_accuracy_euclidean_real = accuracy_euclidean_real
        best_k_euclidean_real = k

# Cetak tabel hasil akurasi
df_results_knn_real = pd.DataFrame(results_real)
print("\nTabel Akurasi KNN berdasarkan K dan Metode Distance (dataSiswa):")
display(df_results_knn_real.round(2).astype(str) + '%') # Convert to string and add % sign

# Tentukan K terbaik untuk masing-masing metode distance
print(f"\nK terbaik untuk Minkowski (p=1) adalah K={best_k_manhattan_real} dengan akurasi: {best_accuracy_manhattan_real:.2%}")
print(f"K terbaik untuk Euclidean (p=2) adalah K={best_k_euclidean_real} dengan akurasi: {best_accuracy_euclidean_real:.2%}")


LATIHAN BARU BAGIAN B: KNN dengan Berbagai Nilai K (dataSiswa)
--- Data Siswa telah dinormalisasi untuk KNN ---

Melatih model KNN dengan berbagai nilai K...

Tabel Akurasi KNN berdasarkan K dan Metode Distance (dataSiswa):


,K,Minkowski (p=1),Euclidean (p=2)
0,1%,75.0%,50.0%
1,3%,75.0%,50.0%
2,5%,50.0%,50.0%
3,7%,50.0%,50.0%
4,9%,50.0%,50.0%
5,11%,50.0%,50.0%
6,13%,50.0%,50.0%



K terbaik untuk Minkowski (p=1) adalah K=1 dengan akurasi: 75.00%
K terbaik untuk Euclidean (p=2) adalah K=1 dengan akurasi: 50.00%


#### Bagian C: Visualisasi Perbandingan pada Dataset Siswa (`dataSiswa`)

In [ ]:
# Penjelasan: Bagian ini membuat visualisasi (bar chart dan line plot)
# untuk membandingkan kinerja KNN dengan berbagai K dan metrik distance pada `dataSiswa`.

print("\n" + "="*50)
print("LATIHAN BARU BAGIAN C: Visualisasi Perbandingan (dataSiswa)")
print("="*50)

# 1. Buat bar chart akurasi untuk semua K dengan Minkowski
plt.figure(figsize=(10, 6))
sns.barplot(x='K', y='Minkowski (p=1)', data=df_results_knn_real)
plt.title('Akurasi KNN (Manhattan Distance) vs. K (dataSiswa)')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi (%)')
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--')
plt.savefig('knn_manhattan_dataSiswa_barchart.png')
plt.close()
print("Bar chart akurasi Manhattan distance (dataSiswa) telah disimpan sebagai 'knn_manhattan_dataSiswa_barchart.png'")

# 2. Buat bar chart akurasi untuk semua K dengan Euclidean
plt.figure(figsize=(10, 6))
sns.barplot(x='K', y='Euclidean (p=2)', data=df_results_knn_real)
plt.title('Akurasi KNN (Euclidean Distance) vs. K (dataSiswa)')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi (%)')
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--')
plt.savefig('knn_euclidean_dataSiswa_barchart.png')
plt.close()
print("Bar chart akurasi Euclidean distance (dataSiswa) telah disimpan sebagai 'knn_euclidean_dataSiswa_barchart.png'")

# 3. Buat line plot perbandingan Minkowski vs Euclidean dalam satu grafik
plt.figure(figsize=(12, 7))
plt.plot(df_results_knn_real['K'], df_results_knn_real['Minkowski (p=1)'], marker='o', label='Manhattan (p=1)')
plt.plot(df_results_knn_real['K'], df_results_knn_real['Euclidean (p=2)'], marker='x', label='Euclidean (p=2)')
plt.title('Perbandingan Akurasi KNN: Manhattan vs. Euclidean (dataSiswa)')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi (%)')
plt.xticks(k_values_real) # Set x-ticks to actual K values
plt.ylim(min(df_results_knn_real['Minkowski (p=1)'].min(), df_results_knn_real['Euclidean (p=2)'].min()) - 5, 100)
plt.grid(True, linestyle='--')
plt.legend()
plt.savefig('knn_comparison_dataSiswa_lineplot.png')
plt.close()
print("Line plot perbandingan KNN distance (dataSiswa) telah disimpan sebagai 'knn_comparison_dataSiswa_lineplot.png'")


LATIHAN BARU BAGIAN C: Visualisasi Perbandingan (dataSiswa)
Bar chart akurasi Manhattan distance (dataSiswa) telah disimpan sebagai 'knn_manhattan_dataSiswa_barchart.png'
Bar chart akurasi Euclidean distance (dataSiswa) telah disimpan sebagai 'knn_euclidean_dataSiswa_barchart.png'
Line plot perbandingan KNN distance (dataSiswa) telah disimpan sebagai 'knn_comparison_dataSiswa_lineplot.png'


#### Bagian D: Kesimpulan Latihan (Baru)

In [ ]:
# Penjelasan: Bagian ini mencetak ringkasan hasil dari semua model yang dilatih
# pada dataset siswa (`dataSiswa`) dan memberikan kesimpulan akhir.

print("\n" + "="*50)
print("LATIHAN BARU BAGIAN D: Kesimpulan (dataSiswa)")
print("="*50)

print("\nRingkasan Hasil Analisis Dataset Siswa (dataSiswa - Is_Normal_BMI):\n")
print("--------------------------------------------------")

# Akurasi Decision Tree
print(f"Akurasi Decision Tree: {accuracy_dt_siswa_real:.2%}")

# K terbaik + akurasi terbaik untuk Minkowski
print(f"K terbaik untuk KNN (Manhattan Distance, p=1): K={best_k_manhattan_real} dengan akurasi {best_accuracy_manhattan_real:.2%}")

# K terbaik + akurasi terbaik untuk Euclidean
print(f"K terbaik untuk KNN (Euclidean Distance, p=2): K={best_k_euclidean_real} dengan akurasi {best_accuracy_euclidean_real:.2%}")

print("--------------------------------------------------")

# Kesimpulan: metode mana yang terbaik dan mengapa
overall_best_accuracy_real = 0.0
overall_best_method_real = ""

if accuracy_dt_siswa_real > overall_best_accuracy_real:
    overall_best_accuracy_real = accuracy_dt_siswa_real
    overall_best_method_real = "Decision Tree"

if best_accuracy_manhattan_real > overall_best_accuracy_real:
    overall_best_accuracy_real = best_accuracy_manhattan_real
    overall_best_method_real = f"KNN dengan Manhattan Distance (K={best_k_manhattan_real})"

if best_accuracy_euclidean_real > overall_best_accuracy_real:
    overall_best_accuracy_real = best_accuracy_euclidean_real
    overall_best_method_real = f"KNN dengan Euclidean Distance (K={best_k_euclidean_real})"


print(f"\nKesimpulan Akhir untuk prediksi Is_Normal_BMI pada dataSiswa:")
print(f"Berdasarkan percobaan ini, model terbaik adalah {overall_best_method_real} "
      f"dengan akurasi {overall_best_accuracy_real:.2%}.")
print("Penting untuk dicatat bahwa pemilihan metode terbaik dapat bervariasi "
      "tergantung pada karakteristik dataset dan tujuan spesifik klasifikasi.")
print("Decision Tree menawarkan interpretasi yang jelas, sedangkan KNN fleksibel "
      "dengan pilihan K dan metrik jarak yang dapat disesuaikan untuk performa optimal.")


LATIHAN BARU BAGIAN D: Kesimpulan (dataSiswa)

Ringkasan Hasil Analisis Dataset Siswa (dataSiswa - Is_Normal_BMI):

--------------------------------------------------
Akurasi Decision Tree: 100.00%
K terbaik untuk KNN (Manhattan Distance, p=1): K=1 dengan akurasi 75.00%
K terbaik untuk KNN (Euclidean Distance, p=2): K=1 dengan akurasi 50.00%
--------------------------------------------------

Kesimpulan Akhir untuk prediksi Is_Normal_BMI pada dataSiswa:
Berdasarkan percobaan ini, model terbaik adalah Decision Tree dengan akurasi 100.00%.
Penting untuk dicatat bahwa pemilihan metode terbaik dapat bervariasi tergantung pada karakteristik dataset dan tujuan spesifik klasifikasi.
Decision Tree menawarkan interpretasi yang jelas, sedangkan KNN fleksibel dengan pilihan K dan metrik jarak yang dapat disesuaikan untuk performa optimal.
